# Homestead Quickstart on Colab T4

Serve a Liquid **LFM2.5** small language model through **Homestead** (this project's
OpenAI-compatible local-LLM gateway) on a free Colab T4 GPU, and drive one real
tool-calling round trip against it.

**What this notebook proves:** Homestead's own gateway (or a llama.cpp fallback, if
the gateway build fails) can serve `LiquidAI/LFM2.5-1.2B-Instruct-GGUF` on a free T4,
respond on `/v1/chat/completions`, and correctly call a tool when asked.

**Colab/T4 caveats -- read before running:**
- Free-tier T4 sessions disconnect after roughly 90 minutes idle and cap out after a
  few hours of connected time -- this isn't a notebook to leave running unattended.
- Colab's disk is ephemeral: every fresh run re-clones this repo and re-downloads the
  model (a 1.2B GGUF is a couple hundred MB -- a minute or two, not more).
- T4 is a Turing-generation GPU: no native bf16. Everything here runs fp16/int4,
  which is what the GGUF quantizations already are, so this isn't a limitation in
  practice.
- **Runtime -> Change runtime type -> T4 GPU** before running anything below.

Series: this is notebook **01** of the Homestead Colab showcase series -- see
[`examples/colab/README.md`](./README.md) for **02** (Model Battle Arena) and the
roadmap for 03-07 (framework showdown, speculative decoding, vision agents, and more).


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


## 1. Clone this repo and load the shared helpers

`colab_common.py` (in this same directory) holds the gateway-build-with-fallback
logic shared by every notebook in this series -- see its docstrings for what each
function actually does and why.


In [ ]:
import os
import subprocess
import sys

if not os.path.isdir("/content/homestead"):
    subprocess.run(
        ["git", "clone", "--quiet", "--depth", "1",
         "https://github.com/olanigan/homestead.git",
         "/content/homestead"],
        check=True,
    )

subprocess.run(["pip", "install", "-q", "openai", "huggingface_hub", "requests"], check=True)

sys.path.insert(0, "/content/homestead/examples/colab")
from colab_common import try_build_gateway, serve_model, pick_and_download_gguf


## 2. Try building the real Homestead gateway, fall back to llama.cpp if it fails

This is a genuine attempt, not a demo of failure: if this repo builds cleanly and the
model gets registered, the rest of this notebook talks to Homestead's real gateway
(`homestead provider start`, fixed port `:3030`). If anything in that chain doesn't
work in a fresh Colab VM, `serve_model()` below detects it automatically (an empty
`/v1/models` response) and starts a plain `llama-cpp-python` server instead --
either way you get a real OpenAI-compatible endpoint to talk to.


In [ ]:
gateway_ok = try_build_gateway()
print("Gateway build:", "OK" if gateway_ok else "skipped/failed -- will try it anyway and fall back if unhealthy")


## 3. Download the model

`LiquidAI/LFM2.5-1.2B-Instruct-GGUF` -- Liquid's 1.2B edge-tuned instruct model, a
good balance of "small enough to be interesting" and "reliable enough at tool-calling
to not need debugging on a first run" -- sub-1B models are demonstrably less reliable
at tool-calling (see notebook 02's battle results for exactly how much less).


In [ ]:
MODEL_REPO = "LiquidAI/LFM2.5-1.2B-Instruct-GGUF"
model_path = pick_and_download_gguf(MODEL_REPO)
print(model_path)


## 4. Start serving

`serve_model()` tries the gateway first (if step 2 built it), verifies it actually
reports a healthy, registered model, and falls back to llama.cpp automatically if
not -- see `colab_common.py` for the exact health-check logic.


In [ ]:
proc, base_url, models, used_gateway = serve_model(model_path, gateway_ok)
assert models, (
    "No server came up healthy. Check /content/gateway.log and "
    "/content/llamacpp-8080.log for what went wrong."
)
print(f"Serving via {'Homestead gateway' if used_gateway else 'llama.cpp fallback'} at {base_url}")
print(models)


## 5. One real tool-calling round trip

Ask the model something it can only answer correctly by calling a tool, and confirm
it actually does.


In [ ]:
from openai import OpenAI

client = OpenAI(base_url=base_url, api_key="none")
MODEL_ID = models["data"][0]["id"]  # whichever id the server actually reports
print("Talking to model id:", MODEL_ID)

tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
}]

resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "What's the weather in Lagos right now?"}],
    tools=tools,
)
print(resp.choices[0].message)


## What's next

- **[`02_model_battle_arena.ipynb`](./02_model_battle_arena.ipynb)** -- same serving
  pattern, but sweeps five models (350M through an 8B-A1B MoE, plus one non-Liquid
  competitor) through five real tool-calling tasks and produces a pass-rate
  leaderboard.
- See [`examples/colab/README.md`](./README.md) for the rest of the planned series
  (03-07: framework showdown, speculative-decoding speed, vision agents, thinking vs.
  instant, and a small-model-vs-big-API-model comparison).

Stop the server before closing this notebook so the T4 is free for whatever runs next:


In [ ]:
proc.terminate()
print("Server stopped.")
